# Reconstrução do phantom XRMC com TIGRE

A geometria abaixo foi derivada diretamente dos arquivos `source.dat` e
`detector.dat`, em vez de reutilizar parâmetros do experimento microXplorer.

Mapeamento de coordenadas:

- eixo de rotação XRMC: **Y**;
- eixo axial TIGRE: primeiro eixo do volume (`z`);
- projeções TIGRE: `(ângulo, v, u)`;
- `DSO = 1000 mm`;
- `DSD = 1150 mm`;
- detector: `1024 × 1024`, pixel `0,28 mm`;
- volume físico da cabeça: aproximadamente `240 × 250 × 250 mm`.


In [1]:
from pathlib import Path
import gc

import matplotlib.pyplot as plt
import numpy as np
import tigre
import tigre.algorithms as algs

DATA_FILE = Path(
    "/mnt/d/Iniciacao_cientifica/head_phantom/"
    "imagens_raw/sinoXRMC_corrigido.npz"
)

# safe: ~64 MiB de volume; balanced: ~234 MiB; high: ~458 MiB
RECON_PRESET = "balanced"
FDK_FILTER = "hann"

# Somente para abrir o antigo sinoAula.npz, que não possui ângulos.
OLD_DATA_ANGLE_STEP_DEG = 1.0


In [2]:
with np.load(DATA_FILE) as data:
    keys = set(data.files)

    if "projs_ln" in keys:
        projs_ln = np.asarray(data["projs_ln"], dtype=np.float32)
    elif "sino" in keys:
        # Compatibilidade com o arquivo antigo.
        transmission = np.asarray(data["sino"], dtype=np.float32)
        if transmission.ndim != 3:
            raise ValueError(transmission.shape)

        # Arquivo antigo: (linhas, colunas, projeções)
        transmission = transmission.transpose(2, 0, 1)
        transmission = np.clip(transmission, 1e-6, 1.0)
        projs_ln = -np.log(transmission).astype(np.float32)
        print(
            "AVISO: arquivo antigo sem metadados; confirme manualmente "
            "OLD_DATA_ANGLE_STEP_DEG."
        )
    else:
        raise KeyError(
            f"Esperava 'projs_ln' ou 'sino'. Chaves: {sorted(keys)}"
        )

    if "angles_deg" in keys:
        angles_deg = np.asarray(data["angles_deg"], dtype=np.float32)
    else:
        angles_deg = (
            np.arange(projs_ln.shape[0], dtype=np.float32)
            * np.float32(OLD_DATA_ANGLE_STEP_DEG)
        )

    DSO_mm = float(data["DSO_mm"]) if "DSO_mm" in keys else 1000.0
    DSD_mm = float(data["DSD_mm"]) if "DSD_mm" in keys else 1150.0
    n_detector = (
        np.asarray(data["nDetector"], dtype=np.int32)
        if "nDetector" in keys
        else np.asarray(projs_ln.shape[1:], dtype=np.int32)
    )
    d_detector_mm = (
        np.asarray(data["dDetector_mm"], dtype=np.float32)
        if "dDetector_mm" in keys
        else np.array([0.28, 0.28], dtype=np.float32)
    )

projs_ln = np.ascontiguousarray(projs_ln, dtype=np.float32)
angles_deg = np.ascontiguousarray(angles_deg.reshape(-1), dtype=np.float32)
angles = np.ascontiguousarray(np.deg2rad(angles_deg), dtype=np.float32)

print("Projeções:", projs_ln.shape, projs_ln.dtype)
print("Ângulos:", angles.shape)
print("Intervalo:", float(angles_deg[0]), "a", float(angles_deg[-1]), "graus")


OSError: [Errno 12] Cannot allocate memory

In [ ]:
if projs_ln.ndim != 3:
    raise ValueError(
        "As projeções devem ter formato (ângulo, linha, coluna)."
    )

if projs_ln.shape[0] != angles.size:
    raise ValueError(
        f"{projs_ln.shape[0]} projeções e {angles.size} ângulos."
    )

if tuple(projs_ln.shape[1:]) != tuple(n_detector):
    raise ValueError(
        f"Projeções {projs_ln.shape[1:]} incompatíveis com "
        f"nDetector={tuple(n_detector)}."
    )

if not np.isfinite(projs_ln).all():
    raise ValueError("As projeções contêm NaN ou infinito.")

# Integrais de linha físicas não são negativas.
np.maximum(projs_ln, 0.0, out=projs_ln)

step_deg = float(np.median(np.diff(angles_deg)))
coverage_deg = float(angles_deg[-1] - angles_deg[0] + step_deg)

print("Passo angular:", step_deg, "grau(s)")
print("Cobertura angular:", coverage_deg, "graus")
print("Faixa das integrais:", float(projs_ln.min()), float(projs_ln.max()))

if coverage_deg < 350:
    raise ValueError(
        "A aquisição não cobre uma volta completa. "
        "Não associe 180 projeções de passo 1° a ângulos de passo 2°. "
        "Refaça a aquisição ou forneça o passo angular verdadeiro."
    )


In [ ]:
geo = tigre.geometry(mode="cone")

geo.DSO = np.float32(DSO_mm)
geo.DSD = np.float32(DSD_mm)

geo.nDetector = n_detector.astype(np.int32)
geo.dDetector = d_detector_mm.astype(np.float32)
geo.sDetector = geo.nDetector * geo.dDetector

# Ordem TIGRE: [axial, plano 1, plano 2].
# XRMC: Y é o eixo de rotação; a cabeça ocupa ~240 mm em Y
# e ~250 mm no plano X-Z.
geo.sVoxel = np.array([240.0, 250.0, 250.0], dtype=np.float32)

presets = {
    "safe": np.array([240, 256, 256], dtype=np.int32),
    "balanced": np.array([384, 400, 400], dtype=np.int32),
    "high": np.array([480, 500, 500], dtype=np.int32),
}

if RECON_PRESET not in presets:
    raise ValueError(f"Preset inválido: {RECON_PRESET}")

geo.nVoxel = presets[RECON_PRESET]
geo.dVoxel = geo.sVoxel / geo.nVoxel

geo.offOrigin = np.zeros(3, dtype=np.float32)
geo.offDetector = np.zeros(2, dtype=np.float32)
geo.rotDetector = np.zeros(3, dtype=np.float32)
geo.COR = np.float32(0.0)
geo.accuracy = np.float32(0.5)
geo.mode = "cone"

volume_gib = np.prod(geo.nVoxel) * 4 / 1024**3
projection_gib = projs_ln.nbytes / 1024**3
fov_at_iso = geo.sDetector * geo.DSO / geo.DSD

print(geo)
print("FOV do detector no isocentro:", fov_at_iso, "mm")
print("Memória do volume float32:", f"{volume_gib:.3f} GiB")
print("Memória das projeções:", f"{projection_gib:.3f} GiB")


In [ ]:
# Diagnóstico visual sem copiar o conjunto inteiro.
indices = np.linspace(0, projs_ln.shape[0] - 1, 4, dtype=int)

fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for axis, index in zip(axes.ravel(), indices):
    image = axis.imshow(
        projs_ln[index],
        cmap="gray",
        vmin=0,
        vmax=np.percentile(projs_ln[index], 99.5),
    )
    axis.set_title(f"Integral de linha — {angles_deg[index]:.2f}°")
    axis.axis("off")

fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.75)
plt.show()


In [ ]:
gc.collect()

imgFDK = algs.fdk(
    projs_ln,
    geo,
    angles,
    filter=FDK_FILTER,
    verbose=True,
)

print(
    "FDK:",
    imgFDK.shape,
    imgFDK.dtype,
    "min =", float(imgFDK.min()),
    "max =", float(imgFDK.max()),
    "mean =", float(imgFDK.mean()),
)


In [ ]:
# Cortes centrais nos três planos.
iz, iy, ix = (np.asarray(imgFDK.shape) // 2).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

planes = [
    (imgFDK[iz, :, :], f"Axial — z={iz}"),
    (imgFDK[:, iy, :], f"Coronal — y={iy}"),
    (imgFDK[:, :, ix], f"Sagital — x={ix}"),
]

vmin = np.percentile(imgFDK, 1)
vmax = np.percentile(imgFDK, 99.5)

for axis, (plane, title) in zip(axes, planes):
    axis.imshow(
        plane,
        cmap="gray",
        origin="lower",
        vmin=vmin,
        vmax=vmax,
    )
    axis.set_title(title)
    axis.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
OUTPUT_VOLUME = Path(
    "/mnt/d/Iniciacao_cientifica/head_phantom/volume_FDK_corrigido.npz"
)

np.savez_compressed(
    OUTPUT_VOLUME,
    volume=imgFDK.astype(np.float32),
    sVoxel_mm=geo.sVoxel.astype(np.float32),
    dVoxel_mm=geo.dVoxel.astype(np.float32),
    nVoxel=geo.nVoxel.astype(np.int32),
    DSO_mm=np.float32(DSO_mm),
    DSD_mm=np.float32(DSD_mm),
    filter=np.asarray(FDK_FILTER),
)

print("Volume salvo:", OUTPUT_VOLUME)
